In [1]:
import pandas as pd
import pulp
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#!pip install openpyxl

In [ ]:
# falls bei Ration Katze kein constraint dabei ist trotzdem mitnehmen und
# im späteren Verlauf einfach als keine Restriktion oder als erfüllte Resitriktion
# 

In [3]:
constraint_df = pd.read_excel(
    "Ration Katze.xlsx",
    header=[0, 1],
    nrows=6
)

constraint_df_og=constraint_df.copy()
#das hier wäre alles andere als dynamisch da müssten wir es schauen wie das Format immer sein würde
constraint_df = constraint_df.iloc[:, 3:-1]
constraint_df = constraint_df.drop(constraint_df.index[-3:-1])
constraint_df_round = constraint_df.round(1)

#exclude all the columns which have no contraints
constraint_df = constraint_df.dropna(axis=1, how="any")
constraint_df = constraint_df.set_index(constraint_df.columns[0])

constraint_df.index = ["Tagesbedarf","Maximaler_Wert","Bedarfsdeck","Grundnahrung"]

constraint_df.columns = [
    f"{str(col[0]).strip()} {str(col[1]).strip()}" if pd.notna(col[1]) else str(col[0]).strip()
    for col in constraint_df.columns
]

In [4]:
constraint_df

,Rp [g],Rfe [g],Ca [mg],P [mg],Ca:P Verhältnis,Mg [mg],K [mg],Na [mg],Cl [mg],Fe [mg],...,Vit. B1 [mg],Vit. B2 [mg],Vit. B6 [mg],Vit. B12 [µg],Biotin [µg],Niacin [mg],Pantothensäure [mg],Folsäure [µg],Vit K [mg],Cholin [mg]
Tagesbedarf,12.556305,5.569329,180.00,160.0000,1.125000,24.049375,330.000000,42.276270,59.996863,5.012396,...,0.354412,0.250620,0.151891,1.417647,4.809875,2.506198,0.354412,48.098751,0.063288,159.485331
Maximaler_Wert,52.000000,16.707987,600.00,500.0000,2.000000,72.148126,12000.000000,100.000000,179.990588,15.037188,...,1.772059,1.253099,0.759454,7.088237,24.049375,12.530990,1.772059,240.493753,0.316439,797.426654
Bedarfsdeck,138.097947,143.643875,74.70,120.5625,61.959565,89.066763,81.060606,186.156441,188.343182,97.358626,...,155.666354,77.368187,227.005204,76.464713,74.014397,158.072104,188.481284,0.000000,13.272694,49.158126
Grundnahrung,17.340000,8.000000,134.46,192.9000,0.697045,21.420000,267.500000,78.700000,113.000000,4.880000,...,0.551700,0.193900,0.344800,1.084000,3.560000,3.961600,0.668000,0.000000,0.008400,78.400000


# die Grundnahrung wird als Zahl bei der Gleichung hinzugefügt werden!!!

In [5]:
#ich unterscheide hier mal nicht zwischen Ergänzungsfuttermittler und Einzelfuttermittel
#Spalte BN komisch bei Einzelfuttermittel auch A Spalte mit einem Eintrag kann man sicher ändern
df_2 = pd.read_excel(
    "Database Supplemente.xlsx",
    sheet_name='EFM',
    header= 2,
)

df_3 = pd.read_excel(
    "Database Supplemente.xlsx",
    sheet_name='Einzelfuttermittel',
    header= 2,
)
df_3.rename(columns={'Taurin [mg]/[100 g]':'Taurin [mg]/[100g]'},inplace=True)
#sollte man im Excel einfach ändern

df_2 = df_2.dropna(axis=1, how="all") #delete exmpty columns
df_3 = df_3.dropna(axis=1, how="all")

df_2 = df_2.dropna(subset=["Identifier"]) #delete rows which weren't fully deleted in the doc
df_3 = df_3.dropna(subset=["Identifier"]) #delete rows which weren't fully deleted in the doc



df_2_slim = df_2.iloc[:, 4:-13]
df_3_slim = df_3.iloc[:, 5:-12]

In [6]:
d3_new_cols=list(set(df_2_slim.columns)-set(df_3_slim.columns))
d2_new_cols=list(set(df_3_slim.columns)-set(df_2_slim.columns))

df_3_slim[d3_new_cols]=0
df_2_slim[d2_new_cols]=0

df_supplements = pd.concat([df_2_slim, df_3_slim], ignore_index=True)


In [7]:
constraint_df.columns

Index(['Rp [g]', 'Rfe [g]', 'Ca [mg]', 'P [mg]', 'Ca:P Verhältnis', 'Mg [mg]',
       'K [mg]', 'Na [mg]', 'Cl [mg]', 'Fe [mg]', 'Cu [mg]', 'Zn [mg]',
       'Mn [mg]', 'Jod [µg]', 'Selen [µg]', 'Vit. A [IE]', 'Vit. D3 [µg]',
       'Vit. E [mg]', 'Vit. B1 [mg]', 'Vit. B2 [mg]', 'Vit. B6 [mg]',
       'Vit. B12 [µg]', 'Biotin [µg]', 'Niacin [mg]', 'Pantothensäure [mg]',
       'Folsäure [µg]', 'Vit K [mg]', 'Cholin [mg]'],
      dtype='object')

In [8]:
#build same name structure 
#find intersection 
## 1) alle constraints Nährstoffe müssen auch bei der Produkte Tabelle dabei sein (ist der Fall muss nur teilweise unbenant werden bzw Vitamin E ka) 
## 2) gleichzeitig sind die Spalten von gewissen Nöhrstoffen bei den Suplements egal falls die nicht bei der Katze vorkommen
## -> erstmal unbennen [100g] weg und : statt / und dann einfach die intersection bilden. Vitamin E wird bei der intersection einfach weggehaut
## hier kann man in zukunft auch ein asert einbauen falls gewisse Spalten nicht gefunden werden


# print(df_supplements.columns)
# print(constraint_df.columns)

cleaned_list_supplements = [s.split("]", 1)[0] + "]" if "]" in s else s for s in list(df_supplements.columns)]
df_supplements.columns=cleaned_list_supplements


constraint_df.rename(columns={'Ca:P Verhältnis':'Ca/P-Verhältnis'},inplace=True)
cleaned_list_constraint = list(constraint_df.columns)


relevante_werte = list(set(cleaned_list_supplements).intersection(set(cleaned_list_constraint)))

In [9]:
print(f'Von allen angegebenen Constraints wird nur {set(cleaned_list_constraint)-set(relevante_werte)} nicht berücksichtigt.')

Von allen angegebenen Constraints wird nur {'Vit. E [mg]'} nicht berücksichtigt.



## Modelierung

In [10]:
final_constraint_df = constraint_df[relevante_werte]
Grundnahrung = final_constraint_df.loc["Grundnahrung"]
final_constraint_df = final_constraint_df.drop(["Grundnahrung", "Bedarfsdeck"])


df_supplements = df_supplements.dropna(subset=["Preis (€) pro kg"]) #anscheinend fehlen Preise diese Zeilen werden einfach gelöscht
Preise = df_supplements['Preis (€) pro kg'] #vlt mit nem Identifier?
Empfolene_Dosierung = df_supplements['Empfolene Dosierung (g pro Tag)']
final_df_supplements = df_supplements[['Futtermittel'] + relevante_werte] #Futtermittel als Index?
final_df_supplements = final_df_supplements.fillna(0)

In [21]:
df_supplements

,Futtermittel,Ra [g],Rp [g],Rfe [g],Rfa [g],Ca [mg],P [mg],Ca/P-Verhältnis,Mg [mg],K [mg],...,Niacin [mg],Pantothensäure [mg],Folsäure [µg],Vit C [mg],Vit K [mg],Cholin [mg],Taurin [mg],Preis (€) pro kg,Empfolene Dosierung (g pro Tag),vRp [g]
0,Revital Plus,5.7,10.3,5.8,1.4,1300.0,1000.0,1.300000,1000.0,1300.0,...,50.0,30.0,600.0,300.0,NaN,1000.0,160.0,96.96,NaN,0.0
1,Catfortan,19.5,34.4,4.5,1.7,4490.0,2200.0,2.040909,250.0,2000.0,...,190.0,50.0,4000.0,NaN,NaN,300.0,1000.0,49.50,1,0.0
2,Biotin-plus MSM Tabs,7.5,15.3,2.4,1.6,NaN,NaN,NaN,NaN,NaN,...,NaN,150.0,NaN,NaN,NaN,NaN,NaN,78.50,0.8,0.0
3,Carni-Fortan,18.5,32.9,4.3,3.7,NaN,NaN,NaN,NaN,NaN,...,100.0,NaN,NaN,500.0,NaN,NaN,2500.0,74.00,4.5,0.0
4,ReConvales AntiPhos,2.0,2.5,4.6,0.4,900.0,70.0,12.857143,NaN,100.0,...,NaN,NaN,NaN,NaN,NaN,40.0,138.0,98.61,10,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Magnesiumgluconat,0.0,0.0,0.0,0.0,NaN,NaN,NaN,5900.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,86.50,0.75,NaN
998,Nachtkerzenöl,0.0,0.0,99.9,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,59.00,NaN,NaN
999,Kasein,0.0,95.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,76.33,3.5,NaN
1000,Seealgenmehl,19.9,0.0,0.0,6.4,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,15.96,1.5,NaN


In [11]:
# =========================================================
# Block 0: Constraints & Grundnahrung
# =========================================================

# Nährstoff-Constraints aus final_constraint_df
min_req = final_constraint_df.loc["Tagesbedarf"]        # Series, Index = relevante_werte
max_req = final_constraint_df.loc["Maximaler_Wert"]     # Series, Index = relevante_werte

# Basis-Aufnahme durch Grundnahrung (Series, gleiche Spalten)
base_intake = Grundnahrung


In [12]:
# =========================================================
# Block 1: Supplements aufbereiten
# =========================================================

# Nährstoffspalten in final_df_supplements sicher numerisch machen
final_df_supplements[relevante_werte] = (
    final_df_supplements[relevante_werte]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)


In [13]:
# =========================================================
# Block 2: Nährstoff-Matrix der Supplements
# =========================================================

# Falls es "Verhältnis"-Spalten gibt, die wir nicht benutzen wollen:
nutrient_cols = [
    c for c in final_constraint_df.columns
    if c in final_df_supplements.columns and "Verhältnis" not in c
]

# Matrix: Zeilen = Supplements, Spalten = Nährstoffe
supp_comp = final_df_supplements.set_index("Futtermittel")[nutrient_cols]

# Sicherheitshalber alles finite
supp_comp = supp_comp.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Namen der Supplements
supp_names = list(supp_comp.index)


In [14]:
# =========================================================
# Block 3: Kosten
# =========================================================

# Preise als numerische Werte
costs = pd.to_numeric(Preise, errors='coerce')

# Index auf Futtermittel setzen, damit es zu supp_comp passt
costs.index = df_supplements['Futtermittel']

# Ausschließlich die Supplements, die auch in supp_comp sind
costs = costs.reindex(supp_names)

# NaN/inf-Kosten entfernen
non_finite = ~np.isfinite(costs)
if non_finite.any():
    print("⚠️ Diese Supplements haben keine gültigen Kosten (NaN/inf) und werden entfernt:")
    print(costs[non_finite])

    bad_supps = list(costs[non_finite].index)
    costs = costs[~non_finite]
    supp_comp = supp_comp.drop(index=bad_supps)
    supp_names = list(supp_comp.index)


In [15]:
# =========================================================
# Block 4: Bedarf nach Grundnahrung
# =========================================================

# Auf tatsächlich verwendete Nährstoffe einschränken
min_req = min_req.reindex(nutrient_cols)
max_req = max_req.reindex(nutrient_cols)
base_intake = base_intake.reindex(nutrient_cols)

# Was müssen die Supplements noch liefern?
min_req_supp = (min_req - base_intake).clip(lower=0)
max_req_supp = (max_req - base_intake)

# Warnung, falls Grundnahrung alleine schon über Max
infeasible_nutrients = max_req_supp[max_req_supp < 0]
if len(infeasible_nutrients) > 0:
    print("Achtung, diese Nährstoffe sind mit der Grundnahrung alleine schon über Maximum:")
    print(infeasible_nutrients)


In [16]:
# =========================================================
# Block 5: LP-Modell
# =========================================================

model = pulp.LpProblem("Katzen_Supplement_Optimierung", pulp.LpMinimize)

# Entscheidungsvariablen: Menge jedes Supplements
x = {
    s: pulp.LpVariable(f"x_{s}", lowBound=0)
    for s in supp_names
}

# Zielfunktion: Kosten minimieren
model += pulp.lpSum(costs[s] * x[s] for s in supp_names), "Gesamtkosten"


In [17]:
# =========================================================
# Block 6: Nährstoff-Constraints
# =========================================================

for nutr in nutrient_cols:
    # Beitrag der Supplements zu diesem Nährstoff
    intake_from_supp = pulp.lpSum(
        supp_comp.loc[s, nutr] * x[s] for s in supp_names
    )

    # Mindestbedarf nach Grundnahrung
    if pd.notna(min_req_supp[nutr]) and min_req_supp[nutr] > 0:
        model += intake_from_supp >= min_req_supp[nutr], f"{nutr}_min"

    # Maximalwert (inkl. Grundnahrung)
    if pd.notna(max_req_supp[nutr]):
        model += intake_from_supp <= max_req_supp[nutr], f"{nutr}_max"


In [18]:
# =========================================================
# Block 7: Lösen
# =========================================================

result = model.solve(pulp.PULP_CBC_CMD(msg=True))
print("Status:", pulp.LpStatus[model.status])


Status: Optimal


In [19]:
# =========================================================
# Block 8: Auswertung
# =========================================================

if pulp.LpStatus[model.status] == "Optimal":
    # Lösung einsammeln
    solution = pd.Series({s: x[s].varValue for s in supp_names}, name="Menge")
    solution = solution.reindex(supp_comp.index)  # zur Sicherheit ausrichten

    print("\nOptimale Supplement-Mengen:")
    print(solution[solution > 1e-9].round(6))

    total_cost = pulp.value(model.objective)
    print("\nMinimale tägliche Kosten:", round(total_cost, 4), "€")

    # Nährstoffbilanz
    intake_supp = supp_comp.T.dot(solution)      # Nährstoff -> Summe aus Supplements

    # base_intake ist bereits Series mit Index = nutrient_cols
    total_intake = intake_supp + base_intake

    report = pd.DataFrame({
        "Grundnahrung": base_intake,
        "Supplemente": intake_supp,
        "Gesamt": total_intake,
        "Min_Bedarf": min_req,
        "Max_Wert": max_req,
    })

    print("\nNährstoff-Bilanz (Grundnahrung + Supplements):")
    print(report.round(3))

else:
    print("Keine optimale Lösung gefunden – prüfe Constraints/Grundnahrung.")



Optimale Supplement-Mengen:
Futtermittel
Hepato K                 0.000686
Immustim K               0.000640
Apto Flex                0.000099
Biosel HK                0.000080
Lilly's Pfotenmix-Bar    0.000376
Vital Komplett Cat       0.003481
BarferMin PLUS           0.007469
Bactisel Gel             0.000274
Kaliumcitrat             0.001734
Name: Menge, dtype: float64

Minimale tägliche Kosten: 1.0848 €

Nährstoff-Bilanz (Grundnahrung + Supplements):
                     Grundnahrung  Supplemente   Gesamt  Min_Bedarf   Max_Wert
Zn [mg]                     2.012        2.798    4.810       4.810     14.430
Vit. D3 [µg]                0.132        0.612    0.744       0.430      2.152
Vit. A [IE]               156.400      609.626  766.026     208.407   1042.035
Cu [mg]                     0.053        0.327    0.379       0.301      0.904
Niacin [mg]                 3.962        4.294    8.256       2.506     12.531
Vit. B12 [µg]               1.084        4.379    5.463       1.41

In [22]:
#CODE VERSTEHEN
#WELCHE EINHEIT FÜR DIE SUPPLEMENTS
Grundnahrung

Zn [mg]                  2.012000
Vit. D3 [µg]             0.132000
Vit. A [IE]            156.400000
Cu [mg]                  0.052600
Niacin [mg]              3.961600
Vit. B12 [µg]            1.084000
Rp [g]                  17.340000
Folsäure [µg]            0.000000
Cl [mg]                113.000000
Vit. B1 [mg]             0.551700
Vit K [mg]               0.008400
Ca [mg]                134.460000
Rfe [g]                  8.000000
Mg [mg]                 21.420000
Na [mg]                 78.700000
P [mg]                 192.900000
Cholin [mg]             78.400000
Jod [µg]                 3.800000
K [mg]                 267.500000
Vit. B2 [mg]             0.193900
Selen [µg]               5.320000
Biotin [µg]              3.560000
Fe [mg]                  4.880000
Ca/P-Verhältnis          0.697045
Pantothensäure [mg]      0.668000
Mn [mg]                  0.058600
Vit. B6 [mg]             0.344800
Name: Grundnahrung, dtype: float64